# Olist Project Notebook

1. Load raw files  
2. Clean data with business rules  
3. Export cleaned CSV files  
4. Generate SQL schema  
5. Validate MySQL row counts  
6. Check the `reviews` table carefully  
7. Create a SQL-safe reviews file  
8. Run row-level validation between CSV and MySQL

**Business rule:** blank review comments are valid. They mean the customer gave a score without writing text.

## 1) Import libraries and set project paths


In [17]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import mysql.connector

PROJECT_DIR = Path(r"C:\Users\Suraj\Desktop\projects\Olist_Project")
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
CLEANED_DATA_DIR = PROJECT_DIR / "data" / "cleaned"
SQL_DIR = PROJECT_DIR / "sql"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATA_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": os.getenv("OLIST_DB_PASSWORD", ""),
    "database": "olist_project"
}

print("Raw data folder:", RAW_DATA_DIR)
print("Cleaned data folder:", CLEANED_DATA_DIR)
print("SQL folder:", SQL_DIR)
print("Database:", DB_CONFIG["database"])

Raw data folder: C:\Users\Suraj\Desktop\projects\Olist_Project\data\raw
Cleaned data folder: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned
SQL folder: C:\Users\Suraj\Desktop\projects\Olist_Project\sql
Database: olist_project


## 2) Load all raw datasets

In [18]:
file_map = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "categories": "product_category_name_translation.csv"
}

dfs = {}

for name, filename in file_map.items():
    path = RAW_DATA_DIR / filename
    dfs[name] = pd.read_csv(path)
    print(f"{name:12} -> {dfs[name].shape}")

orders       -> (99441, 8)
order_items  -> (112650, 7)
payments     -> (103886, 5)
reviews      -> (99224, 7)
customers    -> (99441, 5)
sellers      -> (3095, 4)
products     -> (32951, 9)
geolocation  -> (1000163, 5)
categories   -> (71, 2)


## 3) Inspect structure before cleaning

This helps you understand what each table looks like before any transformation.

In [19]:
for name, df in dfs.items():
    print("=" * 80)
    print(name.upper())
    print(df.info())
    print(df.head(2))

ORDERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   

  order_status order_purchase_

## 4) Clean data with business-aware rules

- blank review comments are valid
- missing dates should stay null
- numeric fields should only be filled when business logic supports it

In [7]:
def standardize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(" ", "_")
                  .str.replace(r"[^a-z0-9_]", "", regex=True)
    )
    return df


def clean_data(name: str, df: pd.DataFrame) -> pd.DataFrame:
    df = df.drop_duplicates().copy()
    df = standardize_column_names(df)

    if name == "orders":
        date_cols = [
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")

    elif name == "reviews":
        date_cols = ["review_creation_date", "review_answer_timestamp"]
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors="coerce")

        if "review_score" in df.columns:
            df["review_score"] = pd.to_numeric(df["review_score"], errors="coerce")

    elif name == "payments":
        numeric_cols = ["payment_sequential", "payment_installments", "payment_value"]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    elif name == "order_items":
        numeric_cols = ["order_item_id", "price", "freight_value"]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        if "shipping_limit_date" in df.columns:
            df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"], errors="coerce")

    elif name == "products":
        numeric_cols = [
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    elif name == "geolocation":
        numeric_cols = ["geolocation_lat", "geolocation_lng"]
        for col in numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

## 5) Apply cleaning and inspect results

This confirms whether the corrected cleaning rules worked as expected.

In [20]:
for name in dfs:
    dfs[name] = clean_data(name, dfs[name])

for name, df in dfs.items():
    print("=" * 80)
    print(name.upper(), "AFTER CLEANING")
    print(df.info())

ORDERS AFTER CLEANING
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB
None
ORDER_ITEMS AFTER CLEANING
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):

## 6) Quick business checks after cleaning

These are sanity checks so you know the tables still make business sense.

In [21]:
print("Orders rows:", len(dfs["orders"]))
print("Payments rows:", len(dfs["payments"]))
print("Reviews rows:", len(dfs["reviews"]))
print()

print("Missing review comment title:", dfs["reviews"]["review_comment_title"].isna().sum())
print("Missing review comment message:", dfs["reviews"]["review_comment_message"].isna().sum())
print()

print("Orders datetime columns:")
print(dfs["orders"].dtypes[dfs["orders"].dtypes.astype(str).str.contains("datetime")])

print()
print("Reviews datetime columns:")
print(dfs["reviews"].dtypes[dfs["reviews"].dtypes.astype(str).str.contains("datetime")])

Orders rows: 99441
Payments rows: 103886
Reviews rows: 99224

Missing review comment title: 87656
Missing review comment message: 58247

Orders datetime columns:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Reviews datetime columns:
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


## 7) Export cleaned datasets to CSV

These files will be used for SQL loading and later validations.

In [22]:
for name, df in dfs.items():
    output_file = CLEANED_DATA_DIR / f"{name}_cleaned.csv"
    df.to_csv(output_file, index=False)
    print("Saved:", output_file)

Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\orders_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\order_items_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\payments_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\reviews_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\customers_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\sellers_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\products_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\geolocation_cleaned.csv
Saved: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\categories_cleaned.csv


## 8) Generate SQL schema from cleaned datasets

This version uses better SQL types for free-text review fields.

In [29]:
def map_sql_type(col_name: str, dtype) -> str:
    dtype = str(dtype).lower()

    if col_name in ["review_comment_title", "review_comment_message"]:
        return "TEXT"
    elif "datetime" in dtype:
        return "DATETIME"
    elif "int" in dtype:
        return "INT"
    elif "float" in dtype:
        return "DECIMAL(10,2)"
    else:
        return "VARCHAR(255)"


primary_keys = {
    "orders": ["order_id"],
    "customers": ["customer_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"]
}

all_tables_sql = ""

for table_name, df in dfs.items():
    columns_sql = []

    for col, dtype in zip(df.columns, df.dtypes):
        sql_type = map_sql_type(col, dtype)
        columns_sql.append(f"{col} {sql_type}")

    pk = primary_keys.get(table_name)
    if pk:
        columns_sql.append(f"PRIMARY KEY ({', '.join(pk)})")

    columns_formatted = ",\n    ".join(columns_sql)

    create_statement = (
        f"CREATE TABLE {table_name} (\n"
        f"    {columns_formatted}\n"
        f");\n"
    )

    all_tables_sql += create_statement + "\n"

schema_path = SQL_DIR / "schema.sql"

with open(schema_path, "w", encoding="utf-8") as f:
    f.write(all_tables_sql)

print("Schema file generated successfully:", schema_path)

Schema file generated successfully: C:\Users\Suraj\Desktop\projects\Olist_Project\sql\schema.sql


## 9) Validate row counts between cleaned CSV files and MySQL tables

In [24]:
def get_connection():
    return mysql.connector.connect(**DB_CONFIG)

files_for_sql_check = {
    "customers": CLEANED_DATA_DIR / "customers_cleaned.csv",
    "orders": CLEANED_DATA_DIR / "orders_cleaned.csv",
    "products": CLEANED_DATA_DIR / "products_cleaned.csv",
    "sellers": CLEANED_DATA_DIR / "sellers_cleaned.csv",
    "order_items": CLEANED_DATA_DIR / "order_items_cleaned.csv",
    "payments": CLEANED_DATA_DIR / "payments_cleaned.csv",
    "reviews": CLEANED_DATA_DIR / "reviews_cleaned.csv"
}

conn = get_connection()
cursor = conn.cursor()

for table_name, file_path in files_for_sql_check.items():
    df = pd.read_csv(file_path)
    csv_count = len(df)

    cursor.execute(f"SELECT COUNT(*) FROM {table_name}")
    sql_count = cursor.fetchone()[0]

    status = "Match" if csv_count == sql_count else "Mismatch"
    print(f"{table_name:12} -> CSV: {csv_count}, SQL: {sql_count}, Status: {status}")

cursor.close()
conn.close()

customers    -> CSV: 99441, SQL: 99441, Status: Match
orders       -> CSV: 99441, SQL: 99441, Status: Match
products     -> CSV: 32951, SQL: 32951, Status: Match
sellers      -> CSV: 3095, SQL: 3095, Status: Match
order_items  -> CSV: 112650, SQL: 112650, Status: Match
payments     -> CSV: 103886, SQL: 103877, Status: Mismatch
reviews      -> CSV: 99224, SQL: 99224, Status: Match


## 10) Review-table checks that matter for business


In [25]:
reviews = pd.read_csv(CLEANED_DATA_DIR / "reviews_cleaned.csv")

print("Total rows:", len(reviews))
print("Unique review_id values:", reviews["review_id"].nunique())
print("Duplicate rows by review_id:", reviews.duplicated(subset=["review_id"]).sum())
print()

duplicate_rows = reviews[reviews.duplicated(subset=["review_id"], keep=False)].sort_values(["review_id", "order_id"])
print("Sample duplicate review_id rows:")
print(duplicate_rows.head(20))

Total rows: 99224
Unique review_id values: 98410
Duplicate rows by review_id: 814

Sample duplicate review_id rows:
                              review_id                          order_id  \
29841  00130cbe1f9d422698c812ed8ded1919  04a28263e085d399c97ae49e0b477efa   
46678  00130cbe1f9d422698c812ed8ded1919  dfcdfc43867d1c1381bfaf62d6b9c195   
63193  0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
90677  0115633a9c298b6a98bcbe4eee75345f  78a4201f58af3463bdab842eea4bc801   
57280  0174caf0ee5964646040cd94e15ac95e  74db91e33b4e1fd865356c89a61abf1f   
92876  0174caf0ee5964646040cd94e15ac95e  f93a732712407c02dce5dd5088d0f47b   
54832  017808d29fd1f942d97e50184dfb4c13  8daaa9e99d60fbba579cc1c3e3bfae01   
99167  017808d29fd1f942d97e50184dfb4c13  b1461c8882153b5fe68307c46a506e39   
96080  0254bd905dc677a6078990aad3331a36  331b367bdd766f3d1cf518777317b5d9   
20621  0254bd905dc677a6078990aad3331a36  5bf226cf882c5bf4247f89a97c86f273   
89712  0288d42bef3dfe36930740c9588a57

## 11) Check review text lengths before SQL loading

This confirms whether free-text columns need `TEXT` instead of short `VARCHAR`.

In [26]:
reviews["title_length"] = reviews["review_comment_title"].fillna("").astype(str).str.len()
reviews["comment_length"] = reviews["review_comment_message"].fillna("").astype(str).str.len()

print("Max title length:", reviews["title_length"].max())
print("Max comment length:", reviews["comment_length"].max())
print("Rows with comment length > 255:", (reviews["comment_length"] > 255).sum())

Max title length: 26
Max comment length: 208
Rows with comment length > 255: 0


## 12) Create a SQL-safe reviews file

This step cleans slashes and line breaks for easier SQL loading, but it keeps true null meaning.

In [27]:
reviews_sql_safe = pd.read_csv(CLEANED_DATA_DIR / "reviews_cleaned.csv")

for col in ["review_comment_title", "review_comment_message"]:
    reviews_sql_safe[col] = reviews_sql_safe[col].astype("string")
    reviews_sql_safe[col] = reviews_sql_safe[col].str.replace("\\", "/", regex=False)
    reviews_sql_safe[col] = reviews_sql_safe[col].str.replace("\r", " ", regex=False)
    reviews_sql_safe[col] = reviews_sql_safe[col].str.replace("\n", " ", regex=False)

reviews_sql_safe_path = CLEANED_DATA_DIR / "reviews_sql_safe.csv"
reviews_sql_safe.to_csv(reviews_sql_safe_path, index=False, encoding="utf-8")

print("SQL-safe reviews file created:", reviews_sql_safe_path)

SQL-safe reviews file created: C:\Users\Suraj\Desktop\projects\Olist_Project\data\cleaned\reviews_sql_safe.csv


## 13) Final row-level validation between CSV and MySQL for the reviews table

This normalizes formats before comparison so you do not get fake mismatches.

In [28]:
reviews_csv = pd.read_csv(CLEANED_DATA_DIR / "reviews_sql_safe.csv")
conn = get_connection()
reviews_sql = pd.read_sql("SELECT * FROM reviews", conn)
conn.close()

date_cols = ["review_creation_date", "review_answer_timestamp"]
text_cols = ["review_id", "order_id", "review_comment_title", "review_comment_message"]
numeric_cols = ["review_score"]

for col in date_cols:
    reviews_csv[col] = pd.to_datetime(reviews_csv[col], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")
    reviews_sql[col] = pd.to_datetime(reviews_sql[col], errors="coerce").dt.strftime("%Y-%m-%d %H:%M:%S").fillna("")

for col in text_cols:
    reviews_csv[col] = reviews_csv[col].fillna("").astype(str)
    reviews_sql[col] = reviews_sql[col].fillna("").astype(str)

for col in numeric_cols:
    reviews_csv[col] = pd.to_numeric(reviews_csv[col], errors="coerce").fillna(0).astype(int).astype(str)
    reviews_sql[col] = pd.to_numeric(reviews_sql[col], errors="coerce").fillna(0).astype(int).astype(str)

reviews_csv["row_key"] = (
    reviews_csv["review_id"] + "|" +
    reviews_csv["order_id"] + "|" +
    reviews_csv["review_score"] + "|" +
    reviews_csv["review_comment_title"] + "|" +
    reviews_csv["review_comment_message"] + "|" +
    reviews_csv["review_creation_date"] + "|" +
    reviews_csv["review_answer_timestamp"]
)

reviews_sql["row_key"] = (
    reviews_sql["review_id"] + "|" +
    reviews_sql["order_id"] + "|" +
    reviews_sql["review_score"] + "|" +
    reviews_sql["review_comment_title"] + "|" +
    reviews_sql["review_comment_message"] + "|" +
    reviews_sql["review_creation_date"] + "|" +
    reviews_sql["review_answer_timestamp"]
)

missing_rows = set(reviews_csv["row_key"]) - set(reviews_sql["row_key"])
extra_rows = set(reviews_sql["row_key"]) - set(reviews_csv["row_key"])

print("CSV rows:", len(reviews_csv))
print("SQL rows:", len(reviews_sql))
print("Missing full rows in SQL:", len(missing_rows))
print("Extra full rows in SQL:", len(extra_rows))

if missing_rows:
    print("\\nSample missing row keys:")
    print(list(missing_rows)[:5])

if extra_rows:
    print("\\nSample extra row keys:")
    print(list(extra_rows)[:5])

C:\Users\Suraj\AppData\Local\Temp\ipykernel_9896\3203345198.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews_sql = pd.read_sql("SELECT * FROM reviews", conn)


CSV rows: 99224
SQL rows: 99224
Missing full rows in SQL: 89385
Extra full rows in SQL: 89385
\nSample missing row keys:
['99dd2836021bb99275780a7b10da6402|b24cae8a85f1becbb93b9150bdd41a7b|5|||2017-09-22 00:00:00|2017-09-23 03:46:49', 'a9bda8bfc86dae045ece299cf7ef986e|28cf6655a339bec298464ed6ac5b4550|1||Comprei dois entregaram só um.|2018-03-21 00:00:00|2018-03-22 09:59:01', 'e4e04ca9de0a5fc447f4436df4a504d7|2341e8f8010d9fd0e367ae298fb20ff0|5|||2017-12-05 00:00:00|2017-12-06 02:26:22', '360df94a293698ff75fcd3779bcca3bd|852d2f4d37773bcbc21c8e09a05a4ea5|3||Produto chegou no prazo o problema que veio na frente da capa danificado faltado couro embaixo por sinal falam que couro mas não muito ruim !!! Não recomendo essa loja !!!!|2018-03-16 00:00:00|2018-03-16 11:12:20', 'b197e6d9f6980e5814e29303f9e07bad|a0b393aaf1085966f34beb2d95817462|5|Ótimo!||2018-05-15 00:00:00|2018-05-16 01:19:19']
\nSample extra row keys:
['64a3fa10f7fbd6ad4b1a0653756f9b67|eacbe7948aadfac0292211f2f352fdeb|5|Unknown|Un

## 14) Notes for business interpretation

- Blank `review_comment_title` and `review_comment_message` are valid.
- Keep comment fields nullable.
- Use all review rows for rating analysis.
- Use only non-blank comment rows for text analysis.
- Do not treat duplicate `review_id` values as automatic errors until full-row checks confirm that.